# Evaluation for Grammar

In [1]:
import argparse
import json
import time
from dataclasses import replace
from difflib import SequenceMatcher
from pathlib import Path

import mlflow
from langchain_core.output_parsers import StrOutputParser

from src.llm.factory import get_model            # adjust if your import path differs
from src.config import get_config
from src.tasks import get_task

# Override path for notebook
import sys
sys.path.insert(0, "C:/Users/Silas/Documents/03_Programming/02_Python/01_Studium/localQuill-v2")

In [2]:
CFG_FIELDS = {"model", "temperature", "max_tokens", "collection", "top_k"}

In [3]:
def build_task(style: str, overrides: dict):
    """Real GrammarTask, with config and model swapped for this eval run."""
    cfg = replace(get_config("grammar", style),
                  **{k: v for k, v in overrides.items() if k in CFG_FIELDS})
    task = get_task("grammar", style)
    task.config = cfg
    task.model = get_model("grammar", style, cfg=cfg, provider=overrides.get("provider"))
    task.chain = task.model | StrOutputParser()
    return task, cfg

In [4]:
def run_case(task, text: str, expected: str | None) -> dict:
    prompt = task.build_prompt(text)
    start = time.time()
    try:
        response, error = task.chain.invoke(prompt), None
    except Exception as e:                    # keep the loop alive on one failure
        response, error = "", f"{type(e).__name__}: {e}"
    latency = time.time() - start
    return {
        "input": text,
        "expected": expected,
        "prompt": prompt,
        "response": response,
        "error": error,
        "latency_s": round(latency, 3),
        "prompt_chars": len(prompt),
        "response_chars": len(response),
        "similarity": (SequenceMatcher(None, response.strip(), expected.strip()).ratio()
                       if expected and not error else None),
    }

In [14]:
def evaluate(config_path: Path, cases: list[dict], style: str) -> None:
    overrides = json.loads(config_path.read_text(encoding="utf-8"))
    run_name = overrides.get("name", config_path.stem)
    task, cfg = build_task(style, overrides)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "provider": overrides.get("provider", "env default"),
            "model": cfg.model, "temperature": cfg.temperature,
            "max_tokens": cfg.max_tokens, "style": style,
            "collection": cfg.collection, "top_k": cfg.top_k,
            "n_cases": len(cases), "config_file": config_path.name,
        })
        mlflow.log_text(type(task).template or "", "prompt_template.txt")

        rows = []
        for i, case in enumerate(cases):
            row = run_case(task, case["input"], case.get("expected"))
            rows.append(row)
            mlflow.log_metric("latency_s", row["latency_s"], step=i)
            if row["similarity"] is not None:
                mlflow.log_metric("similarity", row["similarity"], step=i)

        ok = [r for r in rows if not r["error"]]
        sims = [r["similarity"] for r in ok if r["similarity"] is not None]
        mlflow.log_metrics({
            "error_rate": 1 - len(ok) / len(rows),
            "latency_mean_s": sum(r["latency_s"] for r in ok) / max(len(ok), 1),
            "latency_max_s": max((r["latency_s"] for r in ok), default=0),
            "response_chars_mean": sum(r["response_chars"] for r in ok) / max(len(ok), 1),
            **({"similarity_mean": sum(sims) / len(sims)} if sims else {}),
        })
        # One table with model, prompt and response per case.
        mlflow.log_table(data={k: [r[k] for r in rows] for k in rows[0]},
                         artifact_file="results.json")
        first_err = next((r["error"] for r in rows if r["error"]), None)
        print(f"{run_name}: {len(ok)}/{len(rows)} ok" + (f" | first error: {first_err}" if first_err else ""))

In [24]:
from pathlib import Path
import json
import mlflow

base = Path("C:/Users/Silas/Documents/03_Programming/02_Python/01_Studium/localQuill-v2/evaluation/notebooks/eval")
data_path = base / "grammar_input.json"
config_paths = sorted((base / "configs").glob("*.json"))

cases = json.loads(data_path.read_text(encoding="utf-8"))
assert isinstance(cases, list) and "input" in cases[0], "grammar_input.json must be a list of cases"
print(f"{len(cases)} cases, {len(config_paths)} configs")

# Since we do not use the basic URL
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("Test Grammar")

for config in config_paths:
    evaluate(config, cases, "formal")

2026/11/08 22:49:32 INFO mlflow.tracking.fluent: Experiment with name 'Test Grammar' does not exist. Creating a new experiment.


2 cases, 1 configs


[grammar] retrieval skipped: ChromaDB not found at './chroma'.
[grammar] retrieval skipped: ChromaDB not found at './chroma'.


qwen-coder-0.5b-lmstudio: 2/2 ok
🏃 View run qwen-coder-0.5b-lmstudio at: http://127.0.0.1:5001/#/experiments/1/runs/1c49708c4e9f4f418225b109a9bf9455
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


In [25]:
print(mlflow.get_tracking_uri())

http://127.0.0.1:5001


In [17]:
# LM Studio Debugging
import src.llm.factory as f   # adjust the import path if yours differs
print(f.LMSTUDIO_URL)

import httpx
print(httpx.get("http://127.0.0.1:1234/v1/models", timeout=3).status_code)

http://127.0.0.1:1234/v1


In [27]:
# ToDo:
    # Metrics Time, Token, Tokens per Second, etc
        # (Claude) stream_usage = true in factory.py and the changes that come from that
    # Errant metric comparing response with original error prone text.
    # Errant metric comparing response with Online Corrected Text.
    # Do everything in a script, with variable input and configurations.
        # Prepare the experiments. 10 Formal, 10 Fiction. (Add errors. 2 Outline, 2 User Files, 2 Both, 4 Only Style RAGS)

## Errant Framework